# FFIQ — Indian-language voice (IndicF5)

Clones the scripts in `FFIQ_indic/input/*.json` in the approved course voice, in Hindi (later Marathi, Tamil, Kannada, Telugu).
Fully separate from the English course: it only READS `FFIQ_voice/voice_reference_A.wav` and never writes to `FFIQ_voice/`.

How it works: IndicF5 (AI4Bharat, MIT) is an F5 model trained on Indian languages, so this installs the **same F5-TTS 1.1.22
that already works for the English voice** and loads IndicF5's weights into it. No old-version installs, no restarts.

**One-time:** on huggingface.co click **Agree and access** at https://huggingface.co/ai4bharat/IndicF5, make a **Read**
token, and add it in Colab's **key icon (Secrets)** as `HF_TOKEN` with **Notebook access** ON.

Then: **Runtime → Change runtime type → T4 GPU**, and **Runtime → Run all** (just once). Results: `FFIQ_indic/out/`.

In [ ]:
# CELL 1 — Install F5-TTS 1.1.22 (the version that works for the English voice), ~3-5 min
!nvidia-smi -L || echo 'NO GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again.'
# ONE pip call so the transformers<5 pin holds (same lesson as the English notebook)
!pip install -q f5-tts==1.1.22 'transformers<5' soundfile safetensors
import importlib.metadata as md
for p in ('f5-tts', 'torch', 'transformers', 'numpy', 'vocos'):
    try:
        print(f'{p:13}', md.version(p))
    except Exception:
        print(f'{p:13}', 'not installed')
try:
    import f5_tts.infer.utils_infer  # noqa: F401
except Exception as e:
    raise RuntimeError(f'STOP: F5-TTS does not import ({e!r}). Paste this whole output to Claude.')
print('\n✅ F5-TTS ready.')

In [ ]:
# CELL 2 — Hugging Face access (IndicF5 needs the HF_TOKEN secret, see the notes at the top)
from google.colab import userdata
from huggingface_hub import login
try:
    login(userdata.get('HF_TOKEN'))
except Exception as e:
    raise RuntimeError('STOP: add your Hugging Face token as the Colab secret HF_TOKEN '
                       '(key icon on the left, Notebook access ON). ' + repr(e))
print('✅ Hugging Face connected.')

In [ ]:
# CELL 3 — The voice engine: IndicF5 weights inside F5-TTS 1.1.22
# IndicF5 on the proven F5-TTS 1.1.22 package (same engine as the English
# voice): IndicF5 is an F5 "Base" model trained on Indian languages, so we
# build that model with f5-tts and load IndicF5's weights + vocabulary into it.
# No IndicF5 package install (its old numpy/transformers pins break on Colab).
import random
from collections import Counter

import numpy as np
import soundfile as sf
import torch
import torchaudio
from safetensors.torch import load_file
from f5_tts.model import CFM, DiT
from f5_tts.infer.utils_infer import get_tokenizer, load_vocoder, preprocess_ref_audio_text, infer_process


# f5-tts reads the reference with torchaudio.load, which in new torchaudio needs
# FFmpeg libraries (torchcodec). Read WAVs with soundfile instead: no system deps.
def _load_wav(path, *args, **kwargs):
    data, sr = sf.read(path, dtype='float32', always_2d=True)
    return torch.from_numpy(data.T.copy()), sr


torchaudio.load = _load_wav

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
F5_BASE = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
MEL = dict(n_fft=1024, hop_length=256, win_length=1024, n_mel_channels=100,
           target_sample_rate=24000, mel_spec_type='vocos')
SEED = 1019   # same seed as the English voice: same text -> same delivery


def build_model(vocab_path, ckpt_path=None):
    vocab_char_map, vocab_size = get_tokenizer(vocab_path, 'custom')
    model = CFM(transformer=DiT(**F5_BASE, text_num_embeds=vocab_size, mel_dim=MEL['n_mel_channels']),
                mel_spec_kwargs=MEL, odeint_kwargs=dict(method='euler'), vocab_char_map=vocab_char_map)
    if ckpt_path:
        sd = load_file(ckpt_path)
        print('checkpoint key groups:', sorted({k.split('.')[0] for k in sd}))
        # keep only the speech model (drop e.g. a bundled vocoder) and strip its
        # prefix ("ema_model." or similar) so names match f5-tts's CFM
        pref = Counter(k[:k.index('transformer.')] for k in sd if 'transformer.' in k).most_common(1)[0][0]
        sd = {k[len(pref):]: v for k, v in sd.items() if k.startswith(pref)}
        sd = {k: v for k, v in sd.items() if not k.startswith('mel_spec.') and k not in ('initted', 'step')}
        missing, unexpected = model.load_state_dict(sd, strict=False)
        missing = [k for k in missing if not k.startswith('mel_spec.')]
        print(f'weights: {len(sd)} loaded (prefix {pref!r}), {len(missing)} missing, {len(unexpected)} unexpected')
        if missing or unexpected:
            print('  missing e.g.   ', missing[:6])
            print('  unexpected e.g.', unexpected[:6])
            raise RuntimeError('STOP: IndicF5 weights do not fit the model. Paste this whole output to Claude.')
    return model.to(DEVICE).eval(), vocab_char_map


def not_in_vocab(vocab_char_map, text):
    return sorted({c for c in text if c not in vocab_char_map})


def speak(model, vocoder, ref_audio, ref_text, text, out):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    wav, sr, _ = infer_process(ref_audio, ref_text, text, model, vocoder, mel_spec_type='vocos', device=DEVICE)
    sf.write(out, wav, sr)
    return len(wav) / sr
print('✅ Engine code loaded. Device:', DEVICE)

In [ ]:
# CELL 4 — Google Drive, download IndicF5 (~1.4 GB) and load it
from google.colab import drive
drive.mount('/content/drive', force_remount=True)   # fresh view of files your PC just added
import os, glob, json, hashlib, time, datetime
from huggingface_hub import hf_hub_download

INDIC = '/content/drive/MyDrive/FFIQ_indic'                        # this project's own folder
VOICE = '/content/drive/MyDrive/FFIQ_voice/voice_reference_A.wav'   # approved voice, read only
REF_TXT = f'{INDIC}/voice_reference_A.txt'   # its exact words (kept on Drive, off GitHub)
JOBS = sorted(glob.glob(f'{INDIC}/input/*.json'))
print('Scripts waiting:', [os.path.basename(j) for j in JOBS] or 'none')
for path, what in ((VOICE, 'FFIQ_voice/voice_reference_A.wav'), (REF_TXT, 'FFIQ_indic/voice_reference_A.txt')):
    if not os.path.exists(path):
        raise RuntimeError(f'STOP: {what} is missing (tell Claude).')
if not JOBS:
    raise RuntimeError('STOP: no .json in FFIQ_indic/input/ (tell Claude).')

ckpt = hf_hub_download('ai4bharat/IndicF5', 'model.safetensors')
vocab = hf_hub_download('ai4bharat/IndicF5', 'checkpoints/vocab.txt')
model, VOCAB = build_model(vocab, ckpt)
vocoder = load_vocoder(vocoder_name='vocos', is_local=False, device=DEVICE)
print('✅ IndicF5 loaded on', DEVICE)

In [ ]:
# CELL 5 — Clone (resumes after a disconnect: finished clips are kept)
# Voice A always. If you add FFIQ_indic/input/voice_reference_hi.wav + .txt (10-15 s of
# you speaking Hindi + its exact words), a second set is made to compare the accent.
refs = [('voiceA', VOICE, open(REF_TXT, encoding='utf-8').read().strip())]
hi_wav, hi_txt = f'{INDIC}/input/voice_reference_hi.wav', f'{INDIC}/input/voice_reference_hi.txt'
if os.path.exists(hi_wav) and os.path.exists(hi_txt):
    refs.append(('hindi_ref', hi_wav, open(hi_txt, encoding='utf-8').read().strip()))

for tag, wav, txt in refs:
    gap = not_in_vocab(VOCAB, txt)
    print(f'{tag}: reference letters the model does not know: {gap or "none"}')
    ref_audio, ref_text = preprocess_ref_audio_text(wav, txt)
    for job in JOBS:
        name = os.path.basename(job)[:-5]
        scripts = json.load(open(job, encoding='utf-8'))
        out_dir = f'{INDIC}/out/{name}_{tag}'
        os.makedirs(out_dir, exist_ok=True)
        print(f'\n=== {name} / {tag}: {len(scripts)} clips ===')
        for key, text in scripts.items():
            out = f'{out_dir}/{key}.wav'
            if os.path.exists(out):
                print('  have', key); continue
            unknown = not_in_vocab(VOCAB, text)
            if unknown:
                print(f'  note: {key} has letters the model does not know: {unknown}')
            t0 = time.time()
            secs = speak(model, vocoder, ref_audio, ref_text, text, out)
            print(f'  {key}  {secs:.1f}s of audio in {time.time() - t0:.0f}s')
        json.dump({'lesson': name, 'engine': 'indicf5 on f5-tts ' + md.version('f5-tts'),
                   'model': 'ai4bharat/IndicF5', 'reference': os.path.basename(wav),
                   'voice_md5': hashlib.md5(open(wav, 'rb').read()).hexdigest(),
                   'seed': SEED, 'keys': list(scripts),
                   'finished': datetime.datetime.now().isoformat(timespec='seconds')},
                  open(f'{out_dir}/_DONE.json', 'w'), indent=2)

drive.flush_and_unmount()
print('\n✅ Clips saved to Drive: FFIQ_indic/out/  (tell Claude)')